In [1]:
from collections import deque
import random

ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0: return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def is_solvable(start, goal):
    def count_inversions(state):
        flat = [num for row in state for num in row if num != 0]
        inversions = 0
        for i in range(len(flat)):
            for j in range(i + 1, len(flat)):
                if flat[i] > flat[j]: inversions += 1
        return inversions
    return (count_inversions(start) % 2) == (count_inversions(goal) % 2)

def run_blind_blind_search(max_steps=2000):
    states_pool = list(range(9))
    while True:
        random.shuffle(states_pool)
        s1 = tuple(tuple(states_pool[i:i+3]) for i in range(0, 9, 3))
        random.shuffle(states_pool)
        s2 = tuple(tuple(states_pool[i:i+3]) for i in range(0, 9, 3))
        if s1 != s2 and is_solvable(s1, s2):
            break

    print(f"🎲 Ma trận START tự sinh: {s1}")
    print(f"🎲 Ma trận GOAL tự sinh : {s2}\n")

    queue1, queue2 = deque([(s1, [])]), deque([(s2, [])])
    visited1, visited2 = {s1: []}, {s2: []}
    reverse_move = {'L': 'R', 'R': 'L', 'U': 'D', 'D': 'U'}
    steps = 0

    while queue1 and queue2:
        if steps >= max_steps:
            return None, "Vượt quá giới hạn bước duyệt!"

        # Nhánh Xuôi
        c1, p1 = queue1.popleft()
        steps += 1
        if c1 in visited2:
            return p1 + visited2[c1], f"Thành công sau {steps} bước duyệt."

        x, y = find_zero(c1)
        for move, (dx, dy), _ in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                nxt = swap(c1, x, y, nx, ny)
                if nxt not in visited1:
                    visited1[nxt] = p1 + [move]
                    queue1.append((nxt, p1 + [move]))

        # Nhánh Ngược
        c2, p2 = queue2.popleft()
        steps += 1
        if c2 in visited1:
            return visited1[c2] + p2, f"Thành công sau {steps} bước duyệt."

        x, y = find_zero(c2)
        for move, (dx, dy), _ in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                nxt = swap(c2, x, y, nx, ny)
                if nxt not in visited2:
                    new_path_2 = [reverse_move[move]] + p2
                    visited2[nxt] = new_path_2
                    queue2.append((nxt, new_path_2))

    return None, "Không tìm thấy đường đi."

path, msg = run_blind_blind_search()
print(f"Kết quả: {msg}")
print(f"Đường đi (Lộ trình hành động): {path}")

🎲 Ma trận START tự sinh: ((5, 1, 7), (4, 2, 6), (0, 3, 8))
🎲 Ma trận GOAL tự sinh : ((8, 2, 1), (4, 7, 3), (6, 0, 5))

Kết quả: Thành công sau 1208 bước duyệt.
Đường đi (Lộ trình hành động): ['U', 'R', 'R', 'D', 'L', 'U', 'R', 'U', 'L', 'D', 'L', 'U', 'R', 'D', 'L', 'D', 'R', 'U', 'R', 'D', 'L']


2. Nhìn thấy một p

In [2]:
from collections import deque

ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0: return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def run_partial_state_search(partial_start, goal, max_steps=2000):
    flatten_partial = [num for r in partial_start for num in r]
    fixed_nums = [n for n in flatten_partial if n != -1]
    missing_nums = [n for n in range(9) if n not in fixed_nums]
    possible_starts = []

    # Hàm đệ quy sinh ra các cấu hình ma trận hợp lệ từ ô mờ
    def generate_permutations(idx, current_flat):
        if idx == 9:
            possible_starts.append(tuple(tuple(current_flat[i:i+3]) for i in range(0, 9, 3)))
            return
        if flatten_partial[idx] != -1:
            generate_permutations(idx + 1, current_flat + [flatten_partial[idx]])
        else:
            for num in missing_nums:
                if num not in current_flat:
                    generate_permutations(idx + 1, current_flat + [num])

    generate_permutations(0, [])
    print(f"🔍 Tìm thấy {len(possible_starts)} cấu hình xuất phát khả thi từ ma trận mờ.")

    queue = deque()
    visited = set()
    # Thêm toàn bộ các điểm xuất phát hợp lệ vào Queue (Tìm kiếm đa nguồn)
    for st in possible_starts:
        queue.append((st, []))
        visited.add(st)

    steps = 0
    while queue:
        if steps >= max_steps:
            return None, "Vượt quá giới hạn bước duyệt!"

        current, path = queue.popleft()
        steps += 1

        if current == goal:
            return path, f"Tìm thấy đường đi thành công từ cấu hình xuất phát: {path[0] if path else 'Gốc'}!"

        x, y = find_zero(current)
        for move, (dx, dy), _ in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                nxt = swap(current, x, y, nx, ny)
                if nxt not in visited:
                    visited.add(nxt)
                    queue.append((nxt, path + [move]))

    return None, "Không tìm thấy lời giải khả thi cho cấu hình mờ này."

# -1 đại diện cho ô bị ẩn/mờ
PARTIAL_START = ((1, 2, 3), (4, -1, -1), (7, 5, 8))
GOAL_STATE = ((1, 2, 3), (4, 5, 6), (7, 8, 0))

path, msg = run_partial_state_search(PARTIAL_START, GOAL_STATE)
print(f"Kết quả: {msg}")
print(f"Đường đi: {path}")

🔍 Tìm thấy 2 cấu hình xuất phát khả thi từ ma trận mờ.
Kết quả: Tìm thấy đường đi thành công từ cấu hình xuất phát: D!
Đường đi: ['D', 'R']


3. And-Or-Graph Search

In [3]:
import random

ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0: return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def run_and_or_graph_search(start, goal, max_steps=1000):
    memo_plan = {}
    steps_count = [0] # Sử dụng list để bọc biến đếm dùng trong hàm lồng nhau

    def and_or_search(state, visited_branch):
        if steps_count[0] >= max_steps: return "FAILED"
        steps_count[0] += 1

        if state == goal: return []
        if state in visited_branch: return "FAILED"
        if state in memo_plan: return memo_plan[state]

        visited_branch.add(state)
        x, y = find_zero(state)

        for move, (dx, dy), _ in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                intended_state = swap(state, x, y, nx, ny)
                environment_states = [intended_state]

                # Mô phỏng tác động ngẫu nhiên của môi trường (Nút AND)
                for m_side, (dx_s, dy_s), _ in ACTIONS:
                    nx_s, ny_s = x + dx_s, y + dy_s
                    if 0 <= nx_s < 3 and 0 <= ny_s < 3:
                        side_state = swap(state, x, y, nx_s, ny_s)
                        if side_state != intended_state and random.random() < 0.25:
                            environment_states.append(side_state)

                action_valid_for_all_and = True
                sub_plans = {}

                # Nút AND bắt buộc tất cả các nhánh con từ môi trường đều phải thành công
                for and_state in environment_states:
                    res = and_or_search(and_state, visited_branch.copy())
                    if res == "FAILED":
                        action_valid_for_all_and = False
                        break
                    sub_plans[and_state] = res

                if action_valid_for_all_and:
                    plan = [move] + (sub_plans[intended_state] if intended_state in sub_plans else [])
                    memo_plan[state] = plan
                    return plan

        return "FAILED"

    plan_result = and_or_search(start, set())
    if plan_result == "FAILED" or plan_result is None:
        return None, f"Không tìm thấy chiến lược hành động an toàn sau {steps_count[0]} lần xét."
    return plan_result, f"Tìm thấy chiến lược thành công (Tổng số node đã duyệt: {steps_count[0]})."

START_STATE = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
GOAL_STATE = ((1, 2, 3), (4, 5, 6), (7, 8, 0))

plan, msg = run_and_or_graph_search(START_STATE, GOAL_STATE)
print(f"Kết quả: {msg}")
print(f"Chiến lược hành động (Chuỗi kế hoạch): {plan}")

Kết quả: Không tìm thấy chiến lược hành động an toàn sau 1000 lần xét.
Chiến lược hành động (Chuỗi kế hoạch): None
